# 00 — Setup and data

**Read this before running anything.**

## What this track is

Five notebooks that implement the whole Trust Score modelling pipeline from
scratch, in order, with no dependency on the `trust_score_05` package. They talk
to each other only through files in a run directory. Run them **in order**:

| # | Notebook | Produces |
|---|---|---|
| 00 | `00-setup-and-data.ipynb` | the feature matrices, the fraud labels, `manifest.json` |
| 01 | `01-explore-and-validate.ipynb` | `schema.json`, `quality_report.json` |
| 02 | `02-select-features.ipynb` | `ranked_features.csv`, `selected_features.json` |
| 03 | `03-train-and-score.ipynb` | `models/*.joblib`, `scores.parquet`, `metrics.csv` |
| 04 | `04-compare-and-report.ipynb` | `leaderboard.csv`, `report.md`, `figures/` |

## Why it exists alongside the package track

The package track (`notebooks/1.0-exploration` onward) calls
`trust_score_05.ml.sweep.run_sweep(...)` and a hundred features appear. That is
correct for production and opaque for learning. This track puts every step on the
page: the imputation rules are written out, the ranking score is arithmetic you
can read, the top-k metric is a `numpy` slice. When something in the package
surprises you, come here and find the same step in twenty lines.

It is also the fallback. If the package will not import — wrong Python, missing
JVM, a broken deploy — this track still runs, on a laptop, with nothing but
pandas and scikit-learn.

## Data: real or synthetic

Set `USE_SYNTHETIC = False` and point `FEATURE_MATRIX_PATH` and `FRAUD_PATH` at
real parquet to work on the actual population.

Leave it `True` and this notebook generates a synthetic population with fraud
planted in it. The synthetic data is not a toy: it is built so that the pipeline
finds signal, but only if the pipeline is correct. Specifically it contains
features that are pure noise, features that are 95% null, a constant column, an
identifier that looks numeric, and one column that leaks the label. Notebook 02
should exclude the last three and rank the noise low. If it does not, notebook 02
is broken — which is the point of building the data this way.

## What you need installed

`numpy`, `pandas`, `scikit-learn`, `matplotlib`, `pyarrow` (or `fastparquet`),
`joblib`. No Spark. No pyod. Notebook 03 uses whichever of `pyod` /
`hdbscan` / `torch` happen to be present and skips the rest.

In [ ]:
import json
import os
import pathlib
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Every notebook in this track reads and writes here. Nothing else is shared:
# no imports between notebooks, no global state, no kernel that has to stay
# alive. That is what "standalone" means -- you can run notebook 03 tomorrow on
# a different machine as long as this directory came with you.
RUN_DIR = pathlib.Path(os.environ.get("TS05_RUN_DIR", "./_run")).resolve()
RUN_DIR.mkdir(parents=True, exist_ok=True)

SEED = 20260601
rng = np.random.default_rng(SEED)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

print("run directory:", RUN_DIR)
print("contents     :", sorted(p.name for p in RUN_DIR.iterdir()) or "(empty)")

run directory: C:\Users\govind3\Desktop\trust-score-05-feature-DAT-trust-score-05-rebuild\notebooks\0.0-standalone\_run
contents     : (empty)


## Choose your data source

In [2]:
USE_SYNTHETIC = True

FEATURE_MATRIX_PATH = None   # parquet, one row per customer, wide
FRAUD_PATH = None            # parquet, the confirmed-fraud records

# Synthetic-population size. 40k rows runs in a couple of minutes end to end on
# a laptop; 400k is a better stress test and takes about twenty.
N_CUSTOMERS = 40_000
FRAUD_RATE = 0.004           # about 0.4%, which is realistic and uncomfortable

print("synthetic:", USE_SYNTHETIC)

synthetic: True


## Generating the synthetic population

Read this cell rather than skipping it — the structure of the fake data is the
specification for what the rest of the track has to cope with.

**The honest customers.** Behaviour is drawn from heavy-tailed distributions,
because real telecom behaviour is heavy-tailed: most customers change device
once every few years, a few change monthly, and the mean is not a useful summary
of either group. Counts are Poisson with a per-customer rate drawn from a Gamma,
which is the standard way to get overdispersion.

**The fraud.** Not simply "extreme values". Fraud is planted as a *combination*:
a recent device change AND a recent SIM change AND a burst of EnStream identity
lookups AND short tenure. Each of those alone is common and innocent. Only the
conjunction is rare. That is what makes the problem worth a multivariate
detector, and it is why a univariate threshold on any single feature will do
badly here — as it does on the real data.

**The traps.**

- `cust_noise_1` … `cust_noise_8` — pure Gaussian noise. Should rank low.
- `cust_sparse_flag` — 95% null. Should be cut by the null-rate filter.
- `cust_constant_cnt` — zero variance. Should be cut by the variance filter.
- `customer_id_numeric` — an identifier that happens to be an integer. Should be
  excluded by name, not by statistics, because statistically it looks fine.
- `fraud_case_score` — derived from the label. **Must** be excluded. If it
  survives into notebook 03, every model scores near-perfectly and the whole
  exercise is worthless. This is the single most important trap.

In [3]:
def make_synthetic(n_customers: int, fraud_rate: float, rng) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Build a wide customer feature matrix plus a fraud label table."""
    n_fraud = max(int(n_customers * fraud_rate), 50)
    customer_ids = np.array([f"C{i:07d}" for i in range(n_customers)])

    # --- tenure: a mix of long-standing and recently acquired customers -----
    tenure_days = np.clip(
        rng.gamma(shape=2.0, scale=600.0, size=n_customers), 1, 8000
    ).astype(int)

    # --- per-customer event rates, Gamma-Poisson for overdispersion --------
    def counts(mean_rate: float, window_scale: float) -> np.ndarray:
        rate = rng.gamma(shape=1.4, scale=mean_rate / 1.4, size=n_customers)
        return rng.poisson(rate * window_scale)

    frame = pd.DataFrame({"customer_id": customer_ids})
    frame["cust_days_since_first_seen"] = tenure_days
    frame["cust_days_since_current_assignment_start"] = np.minimum(
        tenure_days, rng.integers(1, 900, size=n_customers)
    )
    frame["cust_distinct_account_cnt"] = 1 + rng.poisson(0.35, size=n_customers)
    frame["cust_distinct_msisdn_cnt"] = 1 + rng.poisson(0.5, size=n_customers)

    for family, rate in [
        ("device_change", 0.9),
        ("sim_change", 0.4),
        ("status_change_c", 0.25),
        ("account_activation", 0.12),
        ("account_cancellation", 0.08),
        ("phone_number_change", 0.05),
        ("mno_port", 0.06),
    ]:
        total = counts(rate, 6.0)
        frame[f"cust_{family}_total_cnt"] = total
        for window, scale in [(3, 0.01), (7, 0.02), (30, 0.09), (90, 0.25), (365, 1.0)]:
            windowed = rng.binomial(total, scale)
            frame[f"cust_{family}_cnt_{window}d"] = windowed
            frame[f"cust_{family}_any_{window}d"] = (windowed > 0).astype(int)
        # recency: only defined for customers who ever had the event. Left as
        # NaN otherwise, on purpose -- see finding 4.5 in the feature findings.
        never = total == 0
        recency = rng.integers(1, 2000, size=n_customers).astype(float)
        recency[never] = np.nan
        frame[f"cust_{family}_days_since_last"] = recency
        frame[f"cust_{family}_interval_mean"] = np.where(
            total > 1, rng.gamma(2.0, 90.0, size=n_customers), np.nan
        )

    # --- EnStream API activity ---------------------------------------------
    enstream_total = counts(2.5, 6.0)
    frame["cust_enstream_api_call_total_cnt"] = enstream_total
    for window, scale in [(3, 0.01), (7, 0.03), (30, 0.10), (90, 0.28), (365, 1.0)]:
        frame[f"cust_enstream_api_call_cnt_{window}d"] = rng.binomial(enstream_total, scale)
    frame["cust_industry_distinct_cnt"] = 1 + rng.poisson(0.8, size=n_customers)
    frame["cust_use_case_distinct_cnt"] = 1 + rng.poisson(0.6, size=n_customers)
    frame["cust_service_provider_name_distinct_cnt"] = 1 + rng.poisson(1.2, size=n_customers)
    frame["cust_new_industry_spread_days"] = np.where(
        frame["cust_industry_distinct_cnt"] > 1,
        rng.gamma(2.0, 120.0, size=n_customers),
        0.0,
    )
    frame["cust_event_type_entropy"] = rng.beta(2.0, 3.0, size=n_customers) * 2.5
    frame["cust_cnt_3d_over_30d"] = np.where(
        frame["cust_device_change_cnt_30d"] > 0,
        frame["cust_device_change_cnt_3d"] / frame["cust_device_change_cnt_30d"].clip(lower=1),
        np.nan,
    )

    # --- the traps ---------------------------------------------------------
    for i in range(1, 9):
        frame[f"cust_noise_{i}"] = rng.normal(size=n_customers)
    sparse = rng.normal(size=n_customers)
    sparse[rng.random(n_customers) < 0.95] = np.nan
    frame["cust_sparse_flag"] = sparse
    frame["cust_constant_cnt"] = 0
    frame["customer_id_numeric"] = np.arange(n_customers)
    # ISO strings rather than pandas timestamps on purpose. Parquet stores a
    # timestamp with an explicit unit, and the engines in this environment do
    # not agree on which unit to write, so a nanosecond frame can come back
    # from disk unreadable ("Cannot losslessly cast ... ms to s"). Nothing in
    # this track does date arithmetic on these columns -- they are provenance,
    # not features -- so a string is strictly better than a portability risk.
    frame["reference_ts"] = "2026-06-01T00:00:00"
    frame["source_path"] = "s3://synthetic/training_nonfraud/"

    # --- plant the fraud ---------------------------------------------------
    fraud_idx = rng.choice(n_customers, size=n_fraud, replace=False)
    frame.loc[fraud_idx, "cust_days_since_first_seen"] = rng.integers(5, 200, size=n_fraud)
    frame.loc[fraud_idx, "cust_device_change_cnt_30d"] += rng.integers(1, 4, size=n_fraud)
    frame.loc[fraud_idx, "cust_device_change_any_30d"] = 1
    frame.loc[fraud_idx, "cust_sim_change_cnt_30d"] += rng.integers(1, 3, size=n_fraud)
    frame.loc[fraud_idx, "cust_sim_change_any_30d"] = 1
    frame.loc[fraud_idx, "cust_enstream_api_call_cnt_7d"] += rng.integers(3, 15, size=n_fraud)
    frame.loc[fraud_idx, "cust_enstream_api_call_cnt_30d"] += rng.integers(5, 25, size=n_fraud)
    frame.loc[fraud_idx, "cust_use_case_distinct_cnt"] += rng.integers(1, 4, size=n_fraud)
    frame.loc[fraud_idx, "cust_device_change_days_since_last"] = rng.integers(
        1, 25, size=n_fraud
    )
    frame.loc[fraud_idx, "cust_sim_change_days_since_last"] = rng.integers(1, 25, size=n_fraud)

    # the leakage trap: near-zero for everyone, informative for fraud
    leak = rng.normal(0.0, 0.01, size=n_customers)
    leak[fraud_idx] = 0.9 + rng.normal(0.0, 0.02, size=n_fraud)
    frame["fraud_case_score"] = leak

    fraud = pd.DataFrame(
        {
            "customer_id": customer_ids[fraud_idx],
            "case_key": [f"CASE{i:06d}" for i in range(n_fraud)],
            "fraud_type": rng.choice(
                [
                    "SIM SWAP",
                    "sim_swap",
                    "Account Takeover",
                    "ACCOUNT TAKEOVER",
                    "port out fraud",
                    "Identity Theft",
                    "subscription fraud",
                    "friendly fraud",          # deliberately unmapped
                    "unknown-new-pattern",     # deliberately unmapped
                ],
                size=n_fraud,
            ),
            "fraud_ts": (
                pd.Timestamp("2026-05-01")
                + pd.to_timedelta(rng.integers(0, 30, size=n_fraud), unit="D")
            ).strftime("%Y-%m-%dT%H:%M:%S"),
            "label": 1,
            "source_path": "s3://synthetic/fraud/nbc/",
        }
    )
    # a handful of duplicated cases, so notebook 03 has something to deduplicate
    fraud = pd.concat([fraud, fraud.sample(n=max(n_fraud // 10, 5), random_state=SEED)])
    return frame, fraud.reset_index(drop=True)


if USE_SYNTHETIC:
    matrix, fraud = make_synthetic(N_CUSTOMERS, FRAUD_RATE, rng)
else:
    matrix = pd.read_parquet(FEATURE_MATRIX_PATH)
    fraud = pd.read_parquet(FRAUD_PATH)

print("matrix:", matrix.shape)
print("fraud :", fraud.shape)
matrix.head()

matrix: (40000, 122)
fraud : (176, 6)


,customer_id,cust_days_since_first_seen,cust_days_since_current_assignment_start,cust_distinct_account_cnt,cust_distinct_msisdn_cnt,cust_device_change_total_cnt,cust_device_change_cnt_3d,cust_device_change_any_3d,cust_device_change_cnt_7d,cust_device_change_any_7d,cust_device_change_cnt_30d,cust_device_change_any_30d,cust_device_change_cnt_90d,cust_device_change_any_90d,cust_device_change_cnt_365d,cust_device_change_any_365d,cust_device_change_days_since_last,cust_device_change_interval_mean,cust_sim_change_total_cnt,cust_sim_change_cnt_3d,...,cust_industry_distinct_cnt,cust_use_case_distinct_cnt,cust_service_provider_name_distinct_cnt,cust_new_industry_spread_days,cust_event_type_entropy,cust_cnt_3d_over_30d,cust_noise_1,cust_noise_2,cust_noise_3,cust_noise_4,cust_noise_5,cust_noise_6,cust_noise_7,cust_noise_8,cust_sparse_flag,cust_constant_cnt,customer_id_numeric,reference_ts,source_path,fraud_case_score
0,C0000000,306,306,1,1,4,0,0,0,0,0,0,0,0,4,1,1661.0,164.122442,0,0,...,1,1,3,0.000000,2.042481,NaN,-0.501671,-2.062648,-0.288396,0.759084,-0.092455,0.335949,-0.037194,0.531325,NaN,0,0,2026-06-01T00:00:00,s3://synthetic/training_nonfraud/,0.013077
1,C0000001,363,363,1,2,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,1,0,...,1,2,1,0.000000,0.360232,NaN,0.025145,-0.306051,0.874966,0.950905,-0.150361,-0.033218,-0.187113,0.310813,NaN,0,1,2026-06-01T00:00:00,s3://synthetic/training_nonfraud/,-0.001258
2,C0000002,2557,728,3,1,6,0,0,0,0,0,0,2,1,6,1,22.0,496.531619,1,0,...,3,1,1,186.229578,0.882723,NaN,0.379499,-0.144711,1.944029,0.997878,-0.386375,0.644622,-0.633533,1.297549,NaN,0,2,2026-06-01T00:00:00,s3://synthetic/training_nonfraud/,-0.001597
3,C0000003,661,552,1,2,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,5,0,...,2,1,4,278.917226,0.606476,NaN,1.746935,-0.754533,0.576090,-0.367459,-0.957363,1.438081,-0.922985,0.903813,NaN,0,3,2026-06-01T00:00:00,s3://synthetic/training_nonfraud/,0.018168
4,C0000004,1334,479,2,1,9,0,0,1,1,1,1,1,1,9,1,1230.0,414.334310,1,0,...,2,1,2,341.890093,0.715799,0.0,-0.038248,1.758325,0.136006,-0.332684,-0.297331,0.682101,-0.218622,-0.406139,NaN,0,4,2026-06-01T00:00:00,s3://synthetic/training_nonfraud/,-0.013413


## Split into the three datasets

The package uses `training_nonfraud`, `testing_nonfraud` and `fraud`. Reproduce
that here, and do the decontamination explicitly so you can see the row count
change.

**The order matters.** Split first, then decontaminate. Decontaminating the whole
matrix and then splitting would leak the fraud identities into the split
decision, which is a subtle enough mistake that it is worth doing in the visible
order.

In [4]:
fraud_customers = set(fraud["customer_id"].unique())
print("fraud customers:", len(fraud_customers))

shuffled = matrix.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
cut = int(len(shuffled) * 0.7)

training_nonfraud = shuffled.iloc[:cut].copy()
testing_nonfraud = shuffled.iloc[cut:].copy()

before = len(training_nonfraud)
training_nonfraud = training_nonfraud[
    ~training_nonfraud["customer_id"].isin(fraud_customers)
].copy()
after = len(training_nonfraud)

print(f"training non-fraud : {before} -> {after}  (removed {before - after} contaminated)")

before = len(testing_nonfraud)
testing_nonfraud = testing_nonfraud[
    ~testing_nonfraud["customer_id"].isin(fraud_customers)
].copy()
print(f"testing non-fraud  : {before} -> {len(testing_nonfraud)}")

# the fraud population, joined back to its features
fraud_features = matrix[matrix["customer_id"].isin(fraud_customers)].copy()
fraud_features["label"] = 1
print(f"fraud with features: {len(fraud_features)}")

fraud customers: 160
training non-fraud : 28000 -> 27891  (removed 109 contaminated)
testing non-fraud  : 12000 -> 11949
fraud with features: 160


## Write everything down

Parquet for the frames, JSON for the manifest. The manifest is how notebook 01
knows what exists without guessing; every later notebook reads it first and
fails loudly if a stage it depends on has not run.

In [5]:
import datetime as dt

paths = {}
for name, frame in [
    ("training_nonfraud", training_nonfraud),
    ("testing_nonfraud", testing_nonfraud),
    ("fraud", fraud_features),
    ("fraud_labels", fraud),
]:
    target = RUN_DIR / f"{name}.parquet"
    frame.to_parquet(target, index=False)
    paths[name] = str(target)
    print(f"{name:20s} {len(frame):7d} rows  {frame.shape[1]:4d} cols  -> {target.name}")

manifest = {
    "created": dt.datetime.now().isoformat(timespec="seconds"),
    "notebook": "00-setup-and-data",
    "synthetic": bool(USE_SYNTHETIC),
    "seed": SEED,
    "n_customers": int(len(matrix)),
    "n_fraud_customers": int(len(fraud_customers)),
    "join_key": "customer_id",
    "label_column": "label",
    "case_key_column": "case_key",
    "provenance_columns": ["source_path", "reference_ts"],
    "known_leakage_columns": ["fraud_case_score"],
    "paths": paths,
    "stages_complete": ["00"],
}
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print()
print(json.dumps(manifest, indent=2))

training_nonfraud      27891 rows   122 cols  -> training_nonfraud.parquet
testing_nonfraud       11949 rows   122 cols  -> testing_nonfraud.parquet
fraud                    160 rows   123 cols  -> fraud.parquet
fraud_labels             176 rows     6 cols  -> fraud_labels.parquet

{
  "created": "2026-09-28T17:37:01",
  "notebook": "00-setup-and-data",
  "synthetic": true,
  "seed": 20260601,
  "n_customers": 40000,
  "n_fraud_customers": 160,
  "join_key": "customer_id",
  "label_column": "label",
  "case_key_column": "case_key",
  "provenance_columns": [
    "source_path",
    "reference_ts"
  ],
  "known_leakage_columns": [
    "fraud_case_score"
  ],
  "paths": {
    "training_nonfraud": "C:\\Users\\govind3\\Desktop\\trust-score-05-feature-DAT-trust-score-05-rebuild\\notebooks\\0.0-standalone\\_run\\training_nonfraud.parquet",
    "testing_nonfraud": "C:\\Users\\govind3\\Desktop\\trust-score-05-feature-DAT-trust-score-05-rebuild\\notebooks\\0.0-standalone\\_run\\testing_nonfraud.p

## Sanity check before you move on

If the fraud rate here is zero, or the fraud frame has no feature columns, stop —
everything downstream will produce confident nonsense.

In [6]:
assert len(fraud_features) > 0, "fraud population is empty -- nothing downstream will be valid"
assert "label" in fraud_features.columns
assert not set(training_nonfraud["customer_id"]) & fraud_customers, "contamination remains"

total = len(training_nonfraud) + len(testing_nonfraud) + len(fraud_features)
print(f"total rows across the three datasets: {total}")
print(f"implied fraud rate at evaluation    : {len(fraud_features) / total:.4%}")
print()
print("Stage 00 complete. Open 01-explore-and-validate.ipynb.")

total rows across the three datasets: 40000
implied fraud rate at evaluation    : 0.4000%

Stage 00 complete. Open 01-explore-and-validate.ipynb.
